# Sentinel-2 pansharpening demo

Loads a Sentinel-2 L2A `.SAFE` product, builds a panchromatic band from the 10m bands, and pansharpens the 20m bands up to 10m with four methods: Brovey, IHS, wavelet substitution, and a zero-shot PNN (CNN). Everything here is a thin call into `climempower_downscaling` -- no algorithm code lives in this notebook.

PNN requires `tensorflow` (`pip install climempower-downscaling[pnn]` or the PNN section of `requirements.txt`); the classical methods do not.

In [ ]:
import os
import platform
import sys
from pathlib import Path

# Jupyter kernels launched by an IDE's kernel picker (VS Code, etc.) typically don't
# run conda's `activate.d` scripts, so on Windows GDAL/PROJ never learn where their
# data files and plugins (e.g. the JP2OpenJPEG driver needed to read Sentinel-2's
# .jp2 bands) live -- this surfaces as "not recognized as being in a supported file
# format ... plugin gdal_JP2OpenJPEG.dll is not available". This replicates exactly
# what `conda activate` sets, using the running kernel's own environment root, so it
# works regardless of how the kernel was launched. No-op on Linux/Mac.
if platform.system() == "Windows":
    env_root = Path(sys.prefix)
    gdal_data = env_root / "Library" / "share" / "gdal"
    gdal_plugins = env_root / "Library" / "lib" / "gdalplugins"
    proj_data = env_root / "Library" / "share" / "proj"
    gdal_bin = env_root / "Library" / "bin"

    if gdal_data.exists():
        os.environ["GDAL_DATA"] = str(gdal_data)
    if gdal_plugins.exists():
        os.environ["GDAL_DRIVER_PATH"] = str(gdal_plugins)
    if proj_data.exists():
        os.environ["PROJ_DATA"] = str(proj_data)
        os.environ["PROJ_LIB"] = str(proj_data)  # older proj/rasterio builds look for this name
    if gdal_bin.exists() and str(gdal_bin) not in os.environ.get("PATH", ""):
        os.environ["PATH"] = str(gdal_bin) + os.pathsep + os.environ.get("PATH", "")

In [ ]:
import numpy as np

import climempower_downscaling
from climempower_downscaling import (
    SENTINEL2_BAND_LABELS,
    brovey,
    discover_band_paths,
    extract_safe_archive,
    generate_panchromatic_band,
    get_georeference,
    ihs,
    make_window,
    normalize_to_unit_range,
    open_bands,
    read_band_subset,
    read_geotiff,
    stack_bands,
    wavelet,
    write_geotiff,
)
from climempower_downscaling.pnn import run_pnn
from climempower_downscaling.visualization import make_rgb_composite, plot_band_comparison

## Configuration

Point this at your own data -- either a product zip (set `ZIP_PATH`) or an already-extracted `.SAFE` directory (set `SAFE_DIR` and leave `ZIP_PATH` as `None`).

In [ ]:
ZIP_PATH = Path(r"E:\Downloads\S2B_.zip")  # e.g. Path("data/S2B_MSIL2A_....zip")
#SAFE_DIR = Path("data/S2A_MSIL2A_....SAFE")  # used when ZIP_PATH is None
EXTRACT_DIR = Path("data/extracted")
OUTPUT_DIR = Path("outputs")

# AOI in pixel coordinates, (col_off, row_off, col_stop, row_stop), at 10m resolution.
# The 20m window is derived by halving these (Sentinel-2's 20m bands are half the
# pixel dimensions of the 10m bands over the same ground area).
BBOX_10M = (0, 0, 2000, 2000)
BBOX_20M = tuple(v // 2 for v in BBOX_10M)

# PNN is a zero-shot, per-scene method: it's normally trained fresh on each scene.
# The committed Weights/PNN_model.h5 was fit on one specific tile, so it's a fast
# way to see PNN output in this demo, but won't be as accurate on a very different
# AOI as training fresh would be. Set this to False to always train from scratch.
USE_PRETRAINED_PNN_WEIGHTS = True

# Anchored to the installed package's own location rather than a path relative to
# the notebook's working directory -- the kernel's cwd isn't always the notebook's
# folder (depends on how it was launched), so a relative "../Weights/..." path can
# silently fail .exists() and fall back to retraining from scratch every run.
REPO_ROOT = Path(climempower_downscaling.__file__).resolve().parent.parent
PNN_WEIGHTS_PATH = REPO_ROOT / "Weights" / "PNN_model.h5"
PNN_EPOCHS = 30  # used only when training from scratch; increase for better quality

OUTPUT_DIR.mkdir(exist_ok=True)

## Load and subset the scene

In [ ]:
safe_dir = extract_safe_archive(ZIP_PATH, EXTRACT_DIR) if ZIP_PATH else SAFE_DIR

paths_10m = discover_band_paths(safe_dir, "10m")
paths_20m = discover_band_paths(safe_dir, "20m")

datasets_10m = open_bands({b: paths_10m[b] for b in ("B02", "B03", "B04", "B08")})
datasets_20m = open_bands(
    {b: paths_20m[b] for b in ("B05", "B06", "B07", "B8A", "B11", "B12")}
)

window_10m = make_window(BBOX_10M)
window_20m = make_window(BBOX_20M)

subset_10m = read_band_subset(datasets_10m, window_10m)
subset_20m = read_band_subset(datasets_20m, window_20m)

## Build the panchromatic band and normalize both stacks to [0, 1]

In [ ]:
pan = generate_panchromatic_band(
    subset_10m["B04"], subset_10m["B03"], subset_10m["B02"], subset_10m["B08"]
)
pan = np.expand_dims(pan, -1)
pan_norm, _, _ = normalize_to_unit_range(pan)

hs_stack = stack_bands(subset_20m)
hs_norm, _, _ = normalize_to_unit_range(hs_stack)

pan_norm.shape, hs_norm.shape

## Run each pansharpening method

In [ ]:
results = {
    "Brovey": brovey(pan_norm, hs_norm),
    "IHS": ihs(pan_norm, hs_norm),
    "Wavelet": wavelet(pan_norm, hs_norm),
}

if USE_PRETRAINED_PNN_WEIGHTS and PNN_WEIGHTS_PATH.exists():
    results["PNN"] = run_pnn(pan_norm, hs_norm, weights_path=PNN_WEIGHTS_PATH)
else:
    results["PNN"] = run_pnn(
        pan_norm,
        hs_norm,
        checkpoint_path=OUTPUT_DIR / "pnn_checkpoint.keras",
        epochs=PNN_EPOCHS,
    )

{name: (image.shape, image.dtype) for name, image in results.items()}

## Compare results

In [ ]:
band_20m_names = list(subset_20m.keys())
band_index = 0  # "B05"

plot_band_comparison(
    subset_20m[band_20m_names[band_index]],
    {name: image[:, :, band_index] for name, image in results.items()},
    band_name=band_20m_names[band_index],
)

In [ ]:
original_rgb = make_rgb_composite(
    subset_20m[band_20m_names[0]], subset_20m[band_20m_names[1]], subset_20m[band_20m_names[2]]
)
pansharpened_rgb = {
    name: make_rgb_composite(image[:, :, 0], image[:, :, 1], image[:, :, 2])
    for name, image in results.items()
}

plot_band_comparison(
    original_rgb,
    pansharpened_rgb,
    band_name=f"{band_20m_names[0]}/{band_20m_names[1]}/{band_20m_names[2]} composite",
    normalize=True,
)

## Save a fused result to GeoTIFF and verify the round-trip

In [ ]:
sample_dataset = next(iter(datasets_10m.values()))
transform_10m, crs_10m = get_georeference(sample_dataset, window_10m)

output_band_order = ("B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12")
band_names = [f"{code} {SENTINEL2_BAND_LABELS[code]}" for code in output_band_order]

output_paths = {}
for method_name, fused in results.items():
    # All bands share the same 10m grid at this point: the native 10m bands, plus
    # the 6 twenty-meter bands now pansharpened onto that grid by this method. Pull
    # each band by code into ascending band-number order, regardless of source array.
    band_arrays = dict(subset_10m)
    for i, code in enumerate(band_20m_names):
        band_arrays[code] = fused[:, :, i]
    combined = np.stack([band_arrays[code] for code in output_band_order], axis=-1)

    output_path = OUTPUT_DIR / f"pansharpened_{method_name.lower()}.tiff"
    write_geotiff(output_path, combined, transform_10m, crs_10m, band_names=band_names)

    read_back, meta = read_geotiff(output_path)
    assert np.array_equal(combined, read_back)
    output_paths[method_name] = output_path
    print(f"Wrote and verified {output_path} -- {meta['count']} bands, CRS {meta['crs']}")

print("Band names:", band_names)

In [ ]:
for dataset in list(datasets_10m.values()) + list(datasets_20m.values()):
    dataset.close()